In [1]:
!pip install langchain langchain-community google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is inc

## Sample Documents in Colab

In [2]:
import os

# Create the directory
os.makedirs('company_docs', exist_ok=True)

# File 1: HR Policy
hr_content = """Employee Handbook HR Policies
Vacation Policy: All full-time employees receive 15 days of paid vacation per year.
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy: Employees may work remotely up to 3 days per week. Remote work requires manager approval.

Parental Leave: 12 weeks paid parental leave for primary caregivers. 6 weeks paid leave for secondary caregivers."""

with open('company_docs/hr_policy.txt', 'w') as f:
    f.write(hr_content)

# File 2: IT Policy (Adding an extra file for comprehensive testing)
it_content = """Company IT & Security Policy
Password Policy: Passwords must be at least 12 characters long and changed every 90 days.
Device Security: Never leave your laptop unattended in public places. Disk encryption must be enabled.
Software Requests: All production software tools must be approved by the IT security department before installation."""

with open('company_docs/it_policy.txt', 'w') as f:
    f.write(it_content)

print("Workspace directory 'company_docs/' and files created successfully!")

Workspace directory 'company_docs/' and files created successfully!


## Part 1 - Document Loading & Chunking

In [4]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load all text files from the directory
loader = DirectoryLoader('company_docs/', glob='*.txt', loader_cls=TextLoader)
documents = loader.load()
print(f'Loaded {len(documents)} documents.\n')

# 2. Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=['\n\n', '\n', ' ', '']
)
chunks = text_splitter.split_documents(documents)
print(f'Split into {len(chunks)} distinct chunks.\n')

print('--- Sample Chunks Preview ---')
for i, chunk in enumerate(chunks[:3]):
    print(f'\nChunk ({i+1}):')
    print(chunk.page_content)
    print(f'Length: {len(chunk.page_content)} characters')
    print('-' * 20)

Loaded 2 documents.

Split into 2 distinct chunks.

--- Sample Chunks Preview ---

Chunk (1):
Employee Handbook HR Policies
Vacation Policy: All full-time employees receive 15 days of paid vacation per year.
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy: Employees may work remotely up to 3 days per week. Remote work requires manager approval.

Parental Leave: 12 weeks paid parental leave for primary caregivers. 6 weeks paid leave for secondary caregivers.
Length: 399 characters
--------------------

Chunk (2):
Company IT & Security Policy
Password Policy: Passwords must be at least 12 characters long and changed every 90 days.
Device Security: Never leave your laptop unattended in public places. Disk encryption must be enabled.
Software Requests: All production software tools must be approved by the IT security department before installation.
Length: 338 characters
--------------------


## Part 2 - Simple Retrieval (Keyword Search)

In [5]:
def simple_search(query, chunks, top_k=3):
    query_lower = query.lower()
    query_words = query_lower.split()
    scored_chunks = []

    for chunk in chunks:
        content_lower = chunk.page_content.lower()
        score = 0
        # Count individual keyword occurrences
        for word in query_words:
            score += content_lower.count(word)

        if score > 0:
            scored_chunks.append((score, chunk))

    # Sort descending by score match
    scored_chunks.sort(reverse=True, key=lambda x: x[0])

    # Return just the text chunks up to top_k limit
    return [chunk for score, chunk in scored_chunks[:top_k]]

# Test the retrieval module
query_test = 'What is the vacation policy?'
relevant_chunks = simple_search(query_test, chunks, top_k=2)

print(f"Query: '{query_test}'")
print(f"Found {len(relevant_chunks)} relevant matches:")
for i, chunk in enumerate(relevant_chunks):
    print(f'\n--- Match Rank {i+1} ---')
    print(chunk.page_content)

Query: 'What is the vacation policy?'
Found 2 relevant matches:

--- Match Rank 1 ---
Employee Handbook HR Policies
Vacation Policy: All full-time employees receive 15 days of paid vacation per year.
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy: Employees may work remotely up to 3 days per week. Remote work requires manager approval.

Parental Leave: 12 weeks paid parental leave for primary caregivers. 6 weeks paid leave for secondary caregivers.

--- Match Rank 2 ---
Company IT & Security Policy
Password Policy: Passwords must be at least 12 characters long and changed every 90 days.
Device Security: Never leave your laptop unattended in public places. Disk encryption must be enabled.
Software Requests: All production software tools must be approved by the IT security department before installation.


# Part 3 - The RAG Pipeline with Gemini

In [7]:
import os
from google import genai
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
client = genai.Client()

def rag_query(query, chunks, top_k=2):
    # Step 1: Retrieve context chunks matching keywords
    relevant_chunks = simple_search(query, chunks, top_k)

    if not relevant_chunks:
        return 'No relevant information found in documents.'

    # Step 2: Combine chunk elements into a single context block
    context = '\n\n---\n\n'.join([c.page_content for c in relevant_chunks])

    # Step 3: Engineer the grounding prompt layout
    prompt = f"""You are a helpful assistant. Answer the question using ONLY the context provided below.
If the answer is not explicitly found in the context context, state "I cannot answer this based on the provided context."

Context:
{context}

Question: {query}
Answer:"""

    # Step 4: Generate grounded response using gemini-2.5-flash
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config={'temperature': 0.0}
    )
    return response.text

##Run Evaluation Suite & Compare (With vs. Without RAG

In [8]:
def ask_without_rag(question):
    prompt = f"You are a helpful HR assistant. Answer this question: {question}"
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )
    return response.text

test_questions = [
    'How many vacation days do full-time employees get?',
    'Can employees work from home?',
    'What is the password length rule?',
    'What is the dress code?'
]

print("=== EVALUATING RAG PERFORMANCE ===")
for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {rag_query(q, chunks)}")
    print("-" * 50)

print("\n" + "="*40)
print("=== CONTRASTING PERSPECTIVES (RAG COMPARISON) ===")
print("="*40)
compare_q = 'How many vacation days do employees get?'

print(f"Target Query: {compare_q}\n")
print(f"WITHOUT RAG (Generic Response):\n{ask_without_rag(compare_q)}")
print("\n" + "-"*30)
print(f"WITH RAG (Company Specific Response):\n{rag_query(compare_q, chunks)}")

=== EVALUATING RAG PERFORMANCE ===

Q: How many vacation days do full-time employees get?
A: All full-time employees receive 15 days of paid vacation per year.
--------------------------------------------------

Q: Can employees work from home?
A: Yes, employees may work remotely up to 3 days per week, but it requires manager approval.
--------------------------------------------------

Q: What is the password length rule?
A: Passwords must be at least 12 characters long.
--------------------------------------------------

Q: What is the dress code?
A: I cannot answer this based on the provided context.
--------------------------------------------------

=== CONTRASTING PERSPECTIVES (RAG COMPARISON) ===
Target Query: How many vacation days do employees get?

WITHOUT RAG (Generic Response):
That's a great question, and it's a very common one!

However, as an AI, I don't have access to the specific policies of your company or any particular organization. The number of vacation days an em